In [1]:
# ======================================================================================
# FULL PIPELINE: MERGED DATA (2024+2025) + LIGHTGBM + JSON EXPORT
# ======================================================================================
!pip install -q implicit lightgbm

import gc
import pickle
import json
import numpy as np
import pandas as pd
import polars as pl
from datetime import datetime
from scipy.sparse import csr_matrix
from scipy import sparse
import implicit
import lightgbm as lgb
from collections import defaultdict
import random
import os

# Cấu hình
pl.Config.set_tbl_rows(20)
RANDOM_STATE = 42
USE_GPU = True

# ======================================================================================
# 1. LOAD DATA & MERGE (2024 + 2025)
# ======================================================================================
print("=" * 60)
print("🚀 FULL PIPELINE: MERGE DATA + JSON EXPORT")
print("=" * 60)

print("\n>>> [1/9] Loading Data...")
data_path = '/kaggle/input/dataset'

# --- 1. Load Ground Truth ---
gt_path = f'{data_path}/final_groundtruth.pkl'
with open(gt_path, 'rb') as f:
    gt_data = pickle.load(f)

ground_truth_dict = {}
if isinstance(gt_data, pd.DataFrame):
    gt_data['customer_id'] = gt_data['customer_id'].astype(str)
    first_item = gt_data['item_id'].iloc[0]
    if isinstance(first_item, (list, np.ndarray)):
        ground_truth_dict = dict(zip(gt_data['customer_id'], gt_data['item_id']))
    else:
        gt_data['item_id'] = gt_data['item_id'].astype(str)
        ground_truth_dict = gt_data.groupby('customer_id')['item_id'].apply(list).to_dict()
else:
    ground_truth_dict = {str(k): [str(i) for i in (v if isinstance(v, list) else [v])] 
                         for k, v in gt_data.items()}

print(f"   Total GT users: {len(ground_truth_dict):,}")

# --- 2. Load Transactions 2024 (Parquet) ---
print("   Loading 2024 Parquet transactions...")
df_trans_2024 = pl.read_parquet(f'{data_path}/sales_pers.purchase_history_daily_chunk_*.parquet')
df_trans_2024 = df_trans_2024.rename({
    df_trans_2024.columns[0]: 'timestamp', 
    df_trans_2024.columns[1]: 'user_id', 
    df_trans_2024.columns[2]: 'item_id'
}).select([
    pl.col('user_id').cast(pl.String), 
    pl.col('item_id').cast(pl.String), 
    # Parquet 2024 thường là seconds -> nhân 1000 -> ms
    (pl.col('timestamp') * 1000).cast(pl.Datetime("ms")).alias('timestamp')
])

# --- 3. Load Transactions Jan 2025 (Pickle) ---
print("   Loading Jan 2025 Pickle transactions...")
pkl_path = f'{data_path}/01-2025.pkl'

# Dùng Pandas đọc Pickle rồi chuyển ngay sang Polars để tối ưu RAM
df_2025_pd = pd.read_pickle(pkl_path)
df_2025_pd = df_2025_pd[['user_id', 'item_id', 'timestamp']]

df_trans_2025 = pl.from_pandas(df_2025_pd).select([
    pl.col('user_id').cast(pl.String),
    pl.col('item_id').cast(pl.String),
    # Timestamp Pickle là Int64 (giây) -> nhân 1000 -> ms
    (pl.col('timestamp') * 1000).cast(pl.Datetime("ms")).alias('timestamp')
])

# Xóa Pandas DF ngay lập tức
del df_2025_pd
gc.collect()

# --- 4. Merge Data ---
print("   Merging datasets...")
df_trans = pl.concat([df_trans_2024, df_trans_2025])
df_trans = df_trans.sort("timestamp")

print(f"   Total Transactions (Merged): {df_trans.height:,}")
print(f"   Time range: {df_trans['timestamp'].min()} to {df_trans['timestamp'].max()}")

# Dọn dẹp
del df_trans_2024, df_trans_2025
gc.collect()

# --- 5. Load User Mapping ---
df_user = pl.read_parquet(f'{data_path}/sales_pers.user_chunk_*.parquet')
df_user = df_user.rename({
    df_user.columns[0]: 'customer_id',
    df_user.columns[-2]: 'user_id'
}).select([
    pl.col('user_id').cast(pl.String), 
    pl.col('customer_id').cast(pl.String)
]).unique(subset=['user_id'])

pdf_user = df_user.to_pandas()
cust_to_user = dict(zip(pdf_user['customer_id'], pdf_user['user_id']))
user_to_cust = dict(zip(pdf_user['user_id'], pdf_user['customer_id']))

# ======================================================================================
# 2. BUILD POPULARITY, HISTORY, FREQUENCY & RECENCY
# ======================================================================================
print("\n>>> [2/9] Building Features (Updated Dates)...")

# [CẬP NHẬT] Reference date dời sang cuối tháng 1/2025
reference_date = datetime(2025, 1, 31) 

# Popularity: Lấy trend 3 tháng gần nhất (T11/2024 -> T1/2025)
pop_df = df_trans.filter(pl.col("timestamp") >= datetime(2024, 11, 1))
if pop_df.height < 1000: pop_df = df_trans

item_popularity = {}
for row in pop_df.group_by('item_id').len().iter_rows():
    item_popularity[str(row[0])] = row[1]
    
top_items_df = pop_df.group_by('item_id').len().sort('len', descending=True).head(100)
global_top_items = top_items_df['item_id'].to_list()

# User history
user_history_dict = {}
history_df = df_trans.group_by('user_id').agg(pl.col('item_id').alias('items'))
for row in history_df.iter_rows():
    user_history_dict[row[0]] = list(set(row[1]))

customer_history_dict = {}
for uid, items in user_history_dict.items():
    cid = user_to_cust.get(uid)
    if cid: customer_history_dict[cid] = items

# Purchase Count
user_purchase_count = {uid: len(items) for uid, items in user_history_dict.items()}

# ===== FREQUENCY & RECENCY =====
user_item_freq = defaultdict(lambda: defaultdict(int))
user_item_last = defaultdict(lambda: defaultdict(lambda: datetime(2024, 1, 1)))
user_item_first = defaultdict(lambda: defaultdict(lambda: datetime(2025, 12, 31)))

# Duyệt qua toàn bộ data đã merge
for row in df_trans.iter_rows():
    ts, uid, item_id = row[2], row[0], row[1]
    user_item_freq[uid][item_id] += 1
    if ts > user_item_last[uid][item_id]: user_item_last[uid][item_id] = ts
    if ts < user_item_first[uid][item_id]: user_item_first[uid][item_id] = ts

# Convert to customer-level dicts
cust_item_freq = defaultdict(lambda: defaultdict(int))
cust_item_last = defaultdict(lambda: defaultdict(lambda: datetime(2024, 1, 1)))
cust_item_first = defaultdict(lambda: defaultdict(lambda: datetime(2025, 12, 31)))

for uid, items_map in user_item_freq.items():
    cid = user_to_cust.get(uid)
    if cid:
        for item_id, freq in items_map.items():
            cust_item_freq[cid][item_id] = freq
            cust_item_last[cid][item_id] = user_item_last[uid][item_id]
            cust_item_first[cid][item_id] = user_item_first[uid][item_id]

# Repurchase Rate
item_buyers = defaultdict(set)
item_repeaters = defaultdict(set)
for uid, items_map in user_item_freq.items():
    for item_id, freq in items_map.items():
        item_buyers[item_id].add(uid)
        if freq > 1: item_repeaters[item_id].add(uid)

item_repurchase_rate = {}
for item_id in item_buyers:
    nb = len(item_buyers[item_id])
    nr = len(item_repeaters[item_id])
    item_repurchase_rate[item_id] = nr / nb if nb > 0 else 0

# History for Evaluation filter
hist_for_eval = {}
for cid in ground_truth_dict.keys():
    uid = cust_to_user.get(cid)
    if uid and uid in user_history_dict:
        hist_for_eval[cid] = [str(i) for i in user_history_dict[uid]]
    elif cid in customer_history_dict:
        hist_for_eval[cid] = [str(i) for i in customer_history_dict[cid]]
    else:
        hist_for_eval[cid] = []

gt_str = {k: [str(i) for i in v] for k, v in ground_truth_dict.items()}

# ======================================================================================
# 3. TRAIN ALS (ON FULL DATA)
# ======================================================================================
print("\n>>> [3/9] Training ALS (Full Data)...")

unique_users = df_trans['user_id'].unique().to_list()
unique_items = df_trans['item_id'].unique().to_list()

user_to_idx = {v: k for k, v in enumerate(unique_users)}
idx_to_item = {k: v for k, v in enumerate(unique_items)}
item_to_idx = {v: k for k, v in enumerate(unique_items)}

# Create Matrix
train_df_idx = df_trans.with_columns([
    pl.col("user_id").replace_strict(user_to_idx, default=None).cast(pl.Int32).alias("u_idx"),
    pl.col("item_id").replace_strict(item_to_idx, default=None).cast(pl.Int32).alias("i_idx")
]).drop_nulls()

counts = train_df_idx.group_by(['u_idx', 'i_idx']).len()
rows = counts['u_idx'].to_numpy()
cols = counts['i_idx'].to_numpy()
data = counts['len'].to_numpy().astype(np.float32)

matrix = csr_matrix((data, (rows, cols)), shape=(len(unique_users), len(unique_items)))

# Train ALS
model_als = implicit.als.AlternatingLeastSquares(
    factors=128, regularization=0.1, alpha=40, iterations=15, random_state=RANDOM_STATE
)
model_als.fit(matrix)
print("   ✅ ALS trained")

# ======================================================================================
# 4. COMPUTE ITEM SIMILARITY
# ======================================================================================
print("\n>>> [4/9] Computing Item Similarity...")
item_matrix = matrix.T.tocsr()
norms = sparse.linalg.norm(item_matrix, axis=1)
norms[norms == 0] = 1
item_matrix_normalized = item_matrix.multiply(1.0 / norms.reshape(-1, 1)).tocsr()

ITEM_SIM_TOP_K = 30
item_similarity = {}
n_items = item_matrix.shape[0]

batch_size = 2000
for start in range(0, n_items, batch_size):
    end = min(start + batch_size, n_items)
    batch_sim = (item_matrix_normalized[start:end] @ item_matrix_normalized.T).toarray()
    for i, row in enumerate(batch_sim):
        item_idx = start + i
        row[item_idx] = 0
        top_indices = np.argsort(row)[::-1][:ITEM_SIM_TOP_K]
        item_similarity[item_idx] = [(int(j), float(row[j])) for j in top_indices if row[j] > 0]

print(f"   ✅ Computed for {len(item_similarity):,} items")

# ======================================================================================
# 5. BUILD LTR TRAINING DATA
# ======================================================================================
print("\n>>> [5/9] Building LTR Training Data...")
# Sử dụng 30k users ngẫu nhiên
sample_users = random.sample(list(customer_history_dict.keys()), min(30000, len(customer_history_dict)))

user_items_csr = matrix.tocsr()
ALS_WEIGHT = 1.0
ITEMCF_WEIGHT = 0.5

ltr_rows, ltr_labels, ltr_groups = [], [], []

for cid in sample_users:
    uid = cust_to_user.get(cid)
    u_idx = user_to_idx.get(uid)
    if u_idx is None: continue
    
    hist_items = set(customer_history_dict.get(cid, []))
    gt_items = set(gt_str.get(cid, []))
    
    # 1. Candidate Generation
    item_scores = {}
    
    # ALS
    try:
        # filter=False để cho phép học lại hành vi mua lặp
        a_ids, a_scores = model_als.recommend([u_idx], user_items_csr[[u_idx]], N=50, filter_already_liked_items=True)
        for idx, sc in zip(a_ids[0], a_scores[0]):
            itm = idx_to_item[idx]
            if itm not in hist_items: # Chỉ lọc history nếu muốn predict hàng mới, ở đây ta train
                item_scores[itm] = {'als': float(sc), 'cf': 0.0}
    except: pass
    
    # ItemCF
    u_hist_idxs = user_items_csr[u_idx].indices[:30]
    for idx in u_hist_idxs:
        if idx in item_similarity:
            for sim_idx, sim_sc in item_similarity[idx][:15]:
                itm = idx_to_item.get(sim_idx)
                if itm:
                    if itm not in item_scores: item_scores[itm] = {'als': 0.0, 'cf': 0.0}
                    item_scores[itm]['cf'] += sim_sc
                    
    if len(item_scores) < 10: continue

    # 2. Build Features
    group_size = 0
    u_pur_cnt = user_purchase_count.get(uid, 0)
    
    for item_id, scores in list(item_scores.items())[:100]:
        label = 1 if item_id in gt_items else 0
        
        als_s = scores['als']
        cf_s = scores.get('cf', 0.0)
        pop = item_popularity.get(item_id, 0)
        u_i_freq = cust_item_freq[cid].get(item_id, 0)
        re_rate = item_repurchase_rate.get(item_id, 0)
        
        # Recency calc
        if u_i_freq > 0:
            d_last = (reference_date - cust_item_last[cid][item_id]).days
            d_first = (reference_date - cust_item_first[cid][item_id]).days
            recency = 1.0 / (1.0 + d_last / 30.0)
        else:
            d_last, d_first, recency = 365, 365, 0.0
            
        feats = [
            als_s, cf_s, als_s*ALS_WEIGHT + cf_s*ITEMCF_WEIGHT,
            pop, np.log1p(pop),
            u_pur_cnt, np.log1p(u_pur_cnt),
            1 if als_s > 0 else 0, 1 if cf_s > 0 else 0,
            u_i_freq, np.log1p(u_i_freq),
            1 if u_i_freq > 0 else 0, 1 if u_i_freq > 2 else 0,
            recency, d_last, np.log1p(d_last), d_first,
            re_rate
        ]
        
        ltr_rows.append(feats)
        ltr_labels.append(label)
        group_size += 1
        
    if group_size > 0: ltr_groups.append(group_size)

X_train = np.array(ltr_rows)
y_train = np.array(ltr_labels)
print(f"   Train Samples: {len(X_train):,}, Pos Rate: {y_train.mean()*100:.2f}%")

# ======================================================================================
# 6. TRAIN LIGHTGBM
# ======================================================================================
print("\n>>> [6/9] Training LightGBM...")
train_dset = lgb.Dataset(X_train, label=y_train, group=ltr_groups)
params = {
    'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_at': [10],
    'learning_rate': 0.05, 'num_leaves': 31, 'feature_fraction': 0.8,
    'device': 'gpu', 'gpu_platform_id': 0, 'gpu_device_id': 0,
    'seed': RANDOM_STATE, 'verbose': -1
}
model_lgb = lgb.train(params, train_dset, num_boost_round=200)
print("   ✅ Model Trained")

# ======================================================================================
# 7. PREDICTION
# ======================================================================================
print("\n>>> [7/9] Generating Predictions...")
all_preds = {}
gt_cids = list(ground_truth_dict.keys())
BATCH = 5000

for i in range(0, len(gt_cids), BATCH):
    if i % 20000 == 0: print(f"   Batch {i}...")
    batch_cids = gt_cids[i:i+BATCH]
    
    for cid in batch_cids:
        uid = cust_to_user.get(cid)
        if uid is None and cid in user_to_idx: uid = cid
        u_idx = user_to_idx.get(uid)
        hist = set(customer_history_dict.get(cid, []))
        
        # Cold Start / Error -> Popular
        if u_idx is None:
            all_preds[cid] = [x for x in global_top_items if x not in hist][:10]
            continue
            
        # Get Candidates
        item_scores = {}
        try:
            # ALS
            a_ids, a_scs = model_als.recommend([u_idx], user_items_csr[[u_idx]], N=50, filter_already_liked_items=True)
            for idx, sc in zip(a_ids[0], a_scs[0]):
                itm = idx_to_item[idx]
                if itm not in hist: item_scores[itm] = {'als': float(sc), 'cf': 0.0}
            
            # ItemCF
            u_itms = user_items_csr[u_idx].indices[:30]
            for idx in u_itms:
                if idx in item_similarity:
                    for s_idx, s_sc in item_similarity[idx][:15]:
                        itm = idx_to_item.get(s_idx)
                        if itm and itm not in hist:
                            if itm not in item_scores: item_scores[itm] = {'als': 0.0, 'cf': 0.0}
                            item_scores[itm]['cf'] += s_sc
        except:
             all_preds[cid] = [x for x in global_top_items if x not in hist][:10]
             continue
             
        # Fallback if few candidates
        if len(item_scores) < 10:
            cur_recs = sorted(item_scores.keys(), key=lambda x: -(item_scores[x]['als'] + item_scores[x]['cf']))
            for pop in global_top_items:
                if len(cur_recs) >= 10: break
                if pop not in cur_recs and pop not in hist: cur_recs.append(pop)
            all_preds[cid] = cur_recs[:10]
            continue
            
        # Feature Build
        u_cnt = user_purchase_count.get(uid, 0)
        cands = list(item_scores.keys())[:100]
        feats_batch = []
        
        for itm in cands:
            sc = item_scores[itm]
            als_s, cf_s = sc['als'], sc.get('cf', 0.0)
            pop = item_popularity.get(itm, 0)
            u_i_freq = cust_item_freq[cid].get(itm, 0)
            
            if u_i_freq > 0:
                d_l = (reference_date - cust_item_last[cid][itm]).days
                d_f = (reference_date - cust_item_first[cid][itm]).days
                recency = 1.0 / (1.0 + d_l / 30.0)
            else:
                d_l, d_f, recency = 365, 365, 0.0
                
            re_rate = item_repurchase_rate.get(itm, 0)
            
            row = [
                als_s, cf_s, als_s*ALS_WEIGHT + cf_s*ITEMCF_WEIGHT,
                pop, np.log1p(pop), u_cnt, np.log1p(u_cnt),
                1 if als_s > 0 else 0, 1 if cf_s > 0 else 0,
                u_i_freq, np.log1p(u_i_freq), 1 if u_i_freq > 0 else 0, 1 if u_i_freq > 2 else 0,
                recency, d_l, np.log1p(d_l), d_f, re_rate
            ]
            feats_batch.append(row)
            
        # Predict
        lgb_sc = model_lgb.predict(np.array(feats_batch))
        srt_idx = np.argsort(lgb_sc)[::-1]
        all_preds[cid] = [cands[x] for x in srt_idx[:10]]

# ======================================================================================
# 8. SCORING
# ======================================================================================
# === Hàm tính score ===
def precision_at_k(pred, gt, hist, filter_bought_items=True, K=10): # prediction, ground-truth, history items, candidate items
    precisions = []
    ideal_precs = []
    ncold_start = 0
    cold_start_users = []
    nusers = len(gt.keys())
    for user in gt.keys():
        if (user not in hist) or (user not in pred):
            ncold_start += 1
            cold_start_users.append(user) # THINKING: để giảm cold start có thể tăng khoảng HISTORY
            continue
        gt_items = gt[user]
        relevant_items = set(gt_items)
        if filter_bought_items:
            relevant_items -=set(hist[user])
        # Compute precision@k
        hits = len(set(pred[user][:K]) & relevant_items)
        precisions.append(hits / K)
    return np.mean(precisions), cold_start_users

print("\n>>> [8/9] Calculating Score...")
score, _ = precision_at_k(all_preds, gt_str, hist_for_eval, filter_bought_items=True, K=10)
print(f"\n{'='*50}\nFINAL SCORE: {score:.6f}\n{'='*50}")

# ======================================================================================
# 9. EXPORT JSON (NEW)
# ======================================================================================
print("\n>>> [9/9] Exporting submission.json...")

# 1. Convert to strict string format
final_submission = {str(k): [str(x) for x in v] for k, v in all_preds.items()}

# 2. Save
output_filename = 'submission.json'
with open(output_filename, 'w') as f:
    json.dump(final_submission, f)

print(f"   ✅ Saved {len(final_submission):,} predictions to {output_filename}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 68.0 MB/s eta 0:00:0000:0100:01
🚀 FULL PIPELINE: MERGE DATA + JSON EXPORT

>>> [1/9] Loading Data...
   Total GT users: 644,970
   Loading 2024 Parquet transactions...
   Loading Jan 2025 Pickle transactions...
   Merging datasets...
   Total Transactions (Merged): 39,028,077
   Time range: 2024-01-01 06:44:59 to 2025-01-30 22:26:25

>>> [2/9] Building Features (Updated Dates)...

>>> [3/9] Training ALS (Full Data)...


/usr/local/lib/python3.11/dist-packages/implicit/cpu/als.py:95: RuntimeWarning: Intel MKL BLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()
/usr/local/lib/python3.11/dist-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

   ✅ ALS trained

>>> [4/9] Computing Item Similarity...
   ✅ Computed for 21,095 items

>>> [5/9] Building LTR Training Data...
   Train Samples: 2,359,847, Pos Rate: 0.28%

>>> [6/9] Training LightGBM...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


   ✅ Model Trained

>>> [7/9] Generating Predictions...
   Batch 0...
   Batch 20000...
   Batch 40000...
   Batch 60000...
   Batch 80000...
   Batch 100000...
   Batch 120000...
   Batch 140000...
   Batch 160000...
   Batch 180000...
   Batch 200000...
   Batch 220000...
   Batch 240000...
   Batch 260000...
   Batch 280000...
   Batch 300000...
   Batch 320000...
   Batch 340000...
   Batch 360000...
   Batch 380000...
   Batch 400000...
   Batch 420000...
   Batch 440000...
   Batch 460000...
   Batch 480000...
   Batch 500000...
   Batch 520000...
   Batch 540000...
   Batch 560000...
   Batch 580000...
   Batch 600000...
   Batch 620000...
   Batch 640000...

>>> [8/9] Calculating Score...

FINAL SCORE: 0.022364

>>> [9/9] Exporting submission.json...
   ✅ Saved 644,970 predictions to submission.json
